# Pandas Data Engineering Project

This notebook contains practical Data Engineering scenarios using a single business dataset.  
The exercises cover data ingestion, profiling, filtering, cleaning, transformation, aggregation, dates, ranking, reshaping, validation, and ETL-style processing.

**Domain:** Manufacturing_Production_Analytics

**Input file:** `10_Manufacturing_Production.csv`

**Questions:** 30

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

df = pd.read_csv("10_Manufacturing_Production.csv")
df.head()


## Dataset Information

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())


## Q1. Numeric summary

Create count, mean, minimum and maximum for the main numeric measure.

In [ ]:
print(df["units_produced"].agg(["count","mean","min","max"]))

## Q2. High-value records

Identify records above the 75th percentile of units_produced.

In [ ]:
t=    production_id    plant  ... production_cost quality_grade
0         PR00001  Plant_A  ...        90268.66             C
1         PR00002  Plant_C  ...       235971.53      Rejected
2         PR00003  Plant_B  ...        35893.35             A
3         PR00004  Plant_D  ...       200801.17             A
4         PR00005  Plant_D  ...        97339.38             B
..            ...      ...  ...             ...           ...
377       PR00378  Plant_D  ...       141560.44             B
378       PR00379  Plant_A  ...       129444.00             C
379       PR00380  Plant_B  ...       150411.31             A
380       PR00010  Plant_C  ...        77447.67             A
381       PR00046  Plant_D  ...        24925.14             B

[382 rows x 10 columns]
threshold=df["units_produced"].quantile(.75)
print(df[df["units_produced"]>threshold].sort_values("units_produced",ascending=False).head(20))

## Q3. Category aggregation

Calculate record count, total units_produced, and average defective_units for each production_id.

In [ ]:
print(df.groupby("production_id",dropna=False).agg(count=("units_produced","count"),total=("units_produced","sum"),avg=("defective_units","mean")).sort_values("total",ascending=False))

## Q4. Two-dimensional aggregation

Analyze units_produced by production_id and plant; keep groups with at least 5 records.

In [ ]:
r=df.groupby(["production_id","plant"],dropna=False).agg(count=("units_produced","count"),total=("units_produced","sum")).query("count>=5").sort_values("total",ascending=False); print(r)

## Q5. Ranking

Rank records by the main numeric measure and display the top 15.

In [ ]:
x=df.copy(); x["rank"]=x["units_produced"].rank(method="dense",ascending=False); print(x.sort_values("rank").head(15))

## Q6. Duplicate audit

Identify duplicate records using the first identifier column.

In [ ]:
key="production_id"; print(df[df.duplicated(key,keep=False)].sort_values(key))

## Q7. Duplicate removal

Create a deduplicated dataset using the first identifier column.

In [ ]:
x=df.drop_duplicates("production_id"); print("Before:",len(df),"After:",len(x))

## Q8. Missing-value profile

Calculate null percentage for every column and sort descending.

In [ ]:
print((df.isna().mean()*100).round(2).sort_values(ascending=False))

## Q9. Categorical fill

Fill missing values in a categorical column with Unknown.

In [ ]:
x=df.copy(); x["production_id"]=x["production_id"].fillna("Unknown"); print(x["production_id"].value_counts(dropna=False))

## Q10. Numeric median fill

Fill missing numeric values using the overall median.

In [ ]:
x=df.copy(); x["units_produced"]=x["units_produced"].fillna(x["units_produced"].median()); print(x["units_produced"].isna().sum())

## Q11. String search

Find records where production_id contains the letter a, case-insensitively.

In [ ]:
print(df[df["production_id"].astype("string").str.contains("a",case=False,na=False)].head(30))

## Q12. Sorted analysis

Sort the dataset by units_produced descending and defective_units ascending.

In [ ]:
print(df.sort_values(["units_produced","defective_units"],ascending=[False,True]).head(25))

## Q13. Conditional segmentation

Create a new segment based on units_produced using its median as the threshold.

In [ ]:
x=df.copy(); med=x["units_produced"].median(); x["segment"]=np.where(x["units_produced"]>=med,"Above Median","Below Median"); print(x["segment"].value_counts())

## Q14. Transform

Use groupby transform to calculate each record's group average of the main measure.

In [ ]:
x=df.copy(); x["group_avg"]=x.groupby("production_id")["units_produced"].transform("mean"); print(x[["production_id","units_produced","group_avg"]].head(25))

## Q15. Group share

Calculate each production_id's share of total units_produced.

In [ ]:
g=df.groupby("production_id",dropna=False)["units_produced"].sum(); print((g/g.sum()*100).round(2).sort_values(ascending=False))

## Q16. Pivot report

Create a pivot table showing average units_produced by production_id and plant.

In [ ]:
print(pd.pivot_table(df,index="production_id",columns="plant",values="units_produced",aggfunc="mean"))

## Q17. Melt report

Create a compact wide summary and convert it to long format using melt.

In [ ]:
w=df.groupby("production_id")[["units_produced","defective_units"]].mean().reset_index(); print(w.melt(id_vars="production_id",var_name="metric",value_name="value"))

## Q18. Validation rule

Count records where units_produced is negative or missing.

In [ ]:
print("Invalid:",df["units_produced"].isna().sum()+(df["units_produced"]<0).sum())

## Q19. Outlier detection

Use IQR to identify high outliers in units_produced.

In [ ]:
q1=df["units_produced"].quantile(.25); q3=df["units_produced"].quantile(.75); iqr=q3-q1; print(df[df["units_produced"]>q3+1.5*iqr][["units_produced"]].head(30))

## Q20. Top group

Find the category in production_id with the highest total units_produced.

In [ ]:
print(df.groupby("production_id")["units_produced"].sum().nlargest(1))

## Q21. Bottom group

Find the three categories in production_id with the lowest average units_produced.

In [ ]:
print(df.groupby("production_id")["units_produced"].mean().nsmallest(3))

## Q22. Multiple aggregations

Produce min, max, mean and median of units_produced for each production_id.

In [ ]:
print(df.groupby("production_id")["units_produced"].agg(["min","max","mean","median"]))

## Q23. Apply rule

Use apply to classify units_produced relative to its median.

In [ ]:
x=df.copy(); med=x["units_produced"].median(); x["classification"]=x["units_produced"].apply(lambda v:"High" if pd.notna(v) and v>=med else "Low"); print(x["classification"].value_counts())

## Q24. Map categories

Create a numeric code for the most frequent categories in the first categorical field.

In [ ]:
codes={{v:i for i,v in enumerate(df["production_id"].dropna().unique())}}; x=df.copy(); x["production_id_code"]=x["production_id"].map(codes); print(x[["production_id","production_id_code"]].head(20))

## Q25. ETL filtering

Build an ETL subset containing non-null units_produced, non-null production_id, and values above the median units_produced.

In [ ]:
m=df["units_produced"].median(); x=df[df["units_produced"].notna()&df["production_id"].notna()&(df["units_produced"]>m)].copy(); print(x.shape); print(x.head())

## Q26. Chunk processing

Read the input CSV in chunks of 50 rows and calculate the main numeric total.

In [ ]:
total=0
count=0
for chunk in pd.read_csv("10_Manufacturing_Production.csv",chunksize=50):
    total += chunk["units_produced"].sum()
    count += len(chunk)
print("Rows processed:",count)
print("Total:",total)

## Q27. Final report

Create a final production_id-level report with count, total units_produced, average defective_units, and rank by total units_produced.

In [ ]:
r=df.groupby("production_id",dropna=False).agg(records=("units_produced","count"),total=("units_produced","sum"),average=("defective_units","mean")).sort_values("total",ascending=False); r["rank"]=r["total"].rank(method="dense",ascending=False).astype(int); print(r)

## End of Project

The notebook is ready to execute with the supplied input CSV.